In [2]:
!pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)


In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_curve
)
import xgboost
from xgboost import XGBClassifier

# =========================
# LOAD DATA
# =========================

df = pd.read_csv(r"C:\Users\himan\Desktop\Projects\pathwayIQ\dataset\processed\cleaned_diabetic_data.csv")

X = df.drop(columns=["target"])
y = df["target"]

# =========================
# TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# =========================
# COLUMN TYPES
# =========================

categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
numerical_cols = X_train.select_dtypes(exclude="object").columns.tolist()

# =========================
# PREPROCESSING
# =========================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# =========================
# SCALE_POS_WEIGHT
# =========================

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("scale_pos_weight:", scale_pos_weight)

# =========================
# XGBOOST PIPELINE
# =========================

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=42
    ))
])

# =========================
# TRAIN
# =========================

xgb_pipeline.fit(X_train, y_train)

# =========================
# PREDICT
# =========================

y_pred = xgb_pipeline.predict(X_test)
y_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

# =========================
# EVALUATE
# =========================

print("\nCLASSIFICATION REPORT:\n")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_proba))

# =========================
# BEST F1 THRESHOLD
# =========================

precisions, recalls, thresholds = precision_recall_curve(
    y_test,
    y_proba
)

f1_scores = 2 * (precisions * recalls) / (
    precisions + recalls + 1e-8
)

best_idx = f1_scores.argmax()

print("\nBest Threshold:", thresholds[best_idx])
print("Best F1:", f1_scores[best_idx])
print("Precision:", precisions[best_idx])
print("Recall:", recalls[best_idx])

scale_pos_weight: 7.959938366718028

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

           0       0.93      0.67      0.78     18082
           1       0.19      0.60      0.28      2271

    accuracy                           0.66     20353
   macro avg       0.56      0.64      0.53     20353
weighted avg       0.85      0.66      0.72     20353


ROC-AUC: 0.6902420189526544

Best Threshold: 0.5418138
Best F1: 0.2989512768183894
Precision: 0.21398707715697454
Recall: 0.4958168207837957
